In [1]:
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [2]:
df = pd.read_csv('../raw/ConsumoMasivoFiltro.csv', sep= ';', low_memory=False)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818543 entries, 0 to 3818542
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   factura              object 
 1   cliente              int64  
 2   nombrecliente        object 
 3   tipocliente          object 
 4   fechafactura         object 
 5   referencia           object 
 6   nombrereferencia     object 
 7   cantidad             float64
 8   preciounitario       float64
 9   costounitario        float64
 10  montoventa           float64
 11  montoventapesos      float64
 12  costoventapesos      float64
 13  valordescuento       float64
 14  unidadnegocios       int64  
 15  centrocostos         int64  
 16  troevendedor         float64
 17  troenombrevendedor   object 
 18  grouptat             object 
 19  lineatat             object 
 20  tro_e_marca          object 
 21  obsequio             int64  
 22  ruta                 float64
 23  porcentajedescuento  float64
 24

In [4]:
df[['ruta', 'zona', 'canal', 'centro_costos','centrocostos','unidadnegocios','troevendedor']] = df[['ruta', 'zona', 'canal', 'centro_costos','centrocostos','unidadnegocios','troevendedor',]].astype(object)
df['fechafactura'] = pd.to_datetime(df['fechafactura'])
df['cliente'] = df['cliente'].astype(int)

In [5]:
df = df[df['obsequio'] ==0]
# Reemplazos
df['referencia'] = df['referencia'].str.replace('TRAAAR03EHDJBLK', 'TRAAAR03EHDBLK', regex=False)
df['referencia'] = df['referencia'].str.replace('TRAAR6EHDJBLK', 'TRAAR6EHDR1', regex=False)

# Relleno de nulos
df.loc[df['canal'].isnull(), 'canal'] = "TAT"
df.loc[df['zona_pais'].isnull(), 'zona_pais'] = "ZONA CENTRO"
df.loc[df['ciclo'].isnull(), 'ciclo'] = "NA"
df.loc[df['zona'].isnull(), 'zona'] = "NA"
df.loc[df['porcentajemargen'].isnull(), 'porcentajemargen'] = 0
df.loc[df['porcentajedescuento'].isnull(), 'porcentajedescuento'] = 0
df.loc[df['ruta'].isnull(), 'ruta'] = "9999"
df.loc[df['tro_e_marca'].isnull(), 'tro_e_marca'] = "OTROS"
df.loc[df['lineatat'].isnull(), 'lineatat'] = "OTROS"
df.loc[df['grouptat'].isnull(), 'grouptat'] = "OTROS"
df.loc[df['troenombrevendedor'].isnull(), 'troenombrevendedor'] = "No Aplica"
df.loc[df['troevendedor'].isnull(), 'troevendedor'] = '9999'
df.loc[df['centro_costos'].isnull(), 'centro_costos'] = "Medellín"

In [6]:
df =df[(df['canal']=='TAT') ]

In [8]:
def obtener_moda(serie):
    return serie.mode()[0]

df_zonas = df.groupby('cliente')['zona'].agg(obtener_moda)

In [9]:
df_zonas.columns = ['cliente','zona']

In [10]:
df_zonas = df_zonas.reset_index()

In [11]:
df_zonas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106084 entries, 0 to 106083
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   cliente  106084 non-null  int64 
 1   zona     106084 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.6+ MB


In [12]:
df_zonas = df_zonas[(df_zonas['zona'].str.startswith('1')) |  (df_zonas['zona'].str.startswith('2'))]

In [13]:
df_modificado = pd.merge(df, df_zonas, on = 'cliente', how = 'inner')

In [15]:
df_modificado = df_modificado.drop(columns =['zona_x'])
df_modificado = df_modificado.rename(columns = {'zona_y':'zona'})


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 944178 entries, 0 to 944177
Data columns (total 31 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   factura              944178 non-null  object        
 1   cliente              944178 non-null  int64         
 2   nombrecliente        944178 non-null  object        
 3   tipocliente          944178 non-null  object        
 4   fechafactura         944178 non-null  datetime64[ns]
 5   referencia           944178 non-null  object        
 6   nombrereferencia     944178 non-null  object        
 7   cantidad             944178 non-null  float64       
 8   preciounitario       944178 non-null  float64       
 9   costounitario        944178 non-null  float64       
 10  montoventa           944178 non-null  float64       
 11  montoventapesos      944178 non-null  float64       
 12  costoventapesos      944178 non-null  float64       
 13  valordescuento

In [16]:
df_modificado.describe()

,cliente,fechafactura,cantidad,preciounitario,costounitario,montoventa,montoventapesos,costoventapesos,valordescuento,obsequio,porcentajedescuento,margenpesos,porcentajemargen
count,"944,178.00",944178,"944,178.00","944,178.00","944,178.00","944,178.00","944,178.00","944,178.00","944,178.00","944,178.00","944,178.00","944,178.00","944,178.00"
mean,"364,183,891.32",2025-05-05 17:56:36.440078080,25.56,"4,038.94","1,770.90","43,593.38","43,593.38","19,136.27","1,565.44",0.00,0.01,"24,457.11",0.55
min,"295,436.00",2024-05-03 00:00:00,"-2,880.00",0.00,181.70,"-5,269,495.51","-5,269,495.51","-2,193,623.42","-300,630.15",0.00,0.00,"-3,075,872.09",-22.89
25%,"32,304,581.00",2024-10-30 00:00:00,5.00,882.35,463.17,"16,806.80","16,806.80","7,679.84",0.00,0.00,0.00,"8,686.57",0.47
50%,"70,662,639.00",2025-05-13 00:00:00,12.00,"2,352.94",938.38,"25,210.20","25,210.20","10,623.73",0.00,0.00,0.00,"13,943.53",0.56
75%,"901,624,533.00",2025-11-06 00:00:00,25.00,"3,781.51","1,574.80","39,495.80","39,495.80","17,711.62",0.00,0.00,0.00,"25,440.35",0.63
max,"10,483,221,957.00",2026-05-09 00:00:00,"14,400.00","1,651,818.33","1,022,250.31","36,847,040.40","36,847,040.40","14,843,385.95","5,505,879.60",0.00,0.97,"22,003,654.45",0.81
std,"489,099,350.12",NaN,99.51,"8,293.46","4,134.85","125,544.07","125,544.07","56,972.32","17,935.73",0.00,0.03,"72,421.00",0.15


In [17]:
df_modificado.to_csv('../raw/Antioquia.csv', index = False, sep = ';', encoding = 'utf-8')